<a href="https://colab.research.google.com/github/ted-chang80/AIFFEL_quest_eng/blob/main/LLM_Application/LLM02/Untitled8_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate

import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

In [ ]:
import tensorflow
import numpy
import transformers
import datasets

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)

2.20.0
2.0.2
5.0.0
4.0.0


In [ ]:
# ==========================================
# 1. 데이터셋 및 토크나이저 로드
# ==========================================
print("1. NSMC 데이터셋 및 KLUE-BERT 토크나이저를 로드하는 중...")
# NSMC 데이터셋 로드 (train, test 분할 포함)
train_file = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
test_file = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
dataset = load_dataset("csv", data_files={"train": train_file, "test": test_file}, delimiter="\t", encoding="utf-8")

# klue/bert-base 토크나이저 불러오기
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

1. NSMC 데이터셋 및 KLUE-BERT 토크나이저를 로드하는 중...


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 2. 데이터 전처리 (토큰화)
# ==========================================
def tokenize_function(examples):
    # 문장을 토큰화하고 최대 길이를 128로 제한합니다.
    return tokenizer(examples["document"], truncation=True, max_length=128)

print("2. 데이터셋 토큰화 진행 중...")

# `document` 컬럼에 None 값이 있거나 문자열이 아닌 경우를 필터링
# NSMC 데이터셋에는 `document`가 비어있는 경우가 있으므로 이를 제거
dataset = dataset.filter(lambda x: x["document"] is not None and len(x["document"]) > 0 and isinstance(x["document"], str), batched=False)

# 속도 향상을 위해 map 함수와 batched=True를 사용합니다.
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 학습용과 평가용 데이터셋 준비
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

2. 데이터셋 토큰화 진행 중...


Filter:   0%|          | 0/150000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 3. 모델 및 평가지표 설정
# ==========================================
print("3. 모델 및 평가지표 설정 중...")
# 이진 분류(긍정/부정)이므로 num_labels=2로 설정합니다.
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 정확도(Accuracy) 측정을 위한 evaluate 라이브러리 활용
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

3. 모델 및 평가지표 설정 중...


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

In [ ]:
# ==========================================
# 4. 학습 하이퍼파라미터 설정 (TrainingArguments)
# ==========================================
training_args = TrainingArguments(
    output_dir="./results",          # 모델 저장 경로
    eval_strategy="epoch",     # 에포크마다 평가 진행
    save_strategy="epoch",           # 에포크마다 모델 저장
    learning_rate=2e-5,              # BERT 모델의 권장 학습률
    per_device_train_batch_size=16,  # 배치 사이즈 (OOM 발생 시 16으로 축소)
    per_device_eval_batch_size=16,
    num_train_epochs=3,              # 총 학습 에포크 수
    weight_decay=0.01,               # 가중치 감쇠(과적합 방지)
    logging_steps=500,               # 500 스텝마다 로그 출력
    load_best_model_at_end=True,     # 학습 종료 후 가장 좋은 모델 로드
    fp16=torch.cuda.is_available(),  # GPU가 지원할 경우 fp16 가속 사용
)

In [ ]:
# ==========================================
# 5. 트레이너 정의 및 학습 시작
# ==========================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# EarlyStoppingCallback 인스턴스 생성
# patience=3: 3 에포크 동안 성능 개선이 없으면 학습 중단
# threshold=0.01: 성능 개선으로 간주할 최소 변화량 (예시)
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[early_stopping_callback], # EarlyStoppingCallback 추가
)

print("4. 파인튜닝 학습을 시작합니다! (시간이 다소 소요될 수 있습니다.)")
trainer.train()

In [ ]:
# ==========================================
# 6. 최종 성능 평가 및 저장
# ==========================================
print("\n5. 최종 테스트 데이터셋 평가:")
eval_results = trainer.evaluate()
print(f"최종 정확도(Accuracy): {eval_results['eval_accuracy']:.4f}")

# 완성된 모델과 토크나이저를 로컬에 저장
model.save_pretrained("./my_nsmc_bert_model")
tokenizer.save_pretrained("./my_nsmc_bert_model")
print("모델 저장이 완료되었습니다!")

### 새로운 하이퍼파라미터로 모델 재학습

모델을 재학습하기 전에, 새로운 학습을 위해 모델을 `klue/bert-base` 사전 학습된 상태로 다시 로드해야 합니다. 그렇지 않으면 기존에 학습된 가중치에서 추가 학습이 진행됩니다.

In [ ]:
# 새로운 학습을 위해 모델을 초기 상태로 다시 로드합니다.
# (이전 모델의 파인튜닝된 가중치가 아닌, klue/bert-base의 사전 학습된 가중치부터 시작)
print("새로운 학습을 위해 모델을 초기화합니다...")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# ==========================================
# 4.1. 변경된 학습 하이퍼파라미터 설정 (TrainingArguments)
# ==========================================

# 예시로 학습률을 1e-5로 낮추고, 배치 사이즈를 16으로 변경해보겠습니다.
# 이 값들은 사용자의 필요에 따라 다시 조정할 수 있습니다.

new_training_args = TrainingArguments(
    output_dir="./results_tuned",  # 변경된 모델 저장 경로
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,            # 변경된 학습률 (예시: 2e-5 -> 1e-5)
    per_device_train_batch_size=16, # 변경된 배치 사이즈 (예시: 32 -> 16)
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

print("새로운 TrainingArguments 설정 완료!")

In [ ]:
# ==========================================
# 6. 최종 성능 평가 및 저장
# ==========================================
print("\n5. 최종 테스트 데이터셋 평가:")
eval_results = trainer.evaluate()
print(f"최종 정확도(Accuracy): {eval_results['eval_accuracy']:.4f}")

# 완성된 모델과 토크나이저를 로컬에 저장
model.save_pretrained("./my_nsmc_bert_model")
tokenizer.save_pretrained("./my_nsmc_bert_model")
print("모델 저장이 완료되었습니다!")

In [ ]:
# ==========================================
# 5.1. 새로운 트레이너 정의 및 학습 시작
# ==========================================

# 데이터 콜레이터는 동일하게 사용합니다.
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


new_trainer = Trainer(
    model=model,
    args=new_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator, # 기존 data_collator 재사용
)

print("5.1. 새로운 하이퍼파라미터로 파인튜닝 학습을 시작합니다! (시간이 다소 소요될 수 있습니다.)")
new_trainer.train()

In [ ]:
# ==========================================
# 6.1. 새로운 모델 최종 성능 평가 및 저장
# ==========================================
print("\n6.1. 새로운 모델의 최종 테스트 데이터셋 평가:")
new_eval_results = new_trainer.evaluate()
print(f"새로운 모델 최종 정확도(Accuracy): {new_eval_results['eval_accuracy']:.4f}")

# 완성된 모델과 토크나이저를 새로운 경로에 저장
new_model_save_path = "./my_nsmc_bert_model_tuned"
model.save_pretrained(new_model_save_path)
tokenizer.save_pretrained(new_model_save_path)
print(f"새로운 모델이 '{new_model_save_path}' 경로에 저장되었습니다!")

### 학습 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

# Trainer의 로그 기록 가져오기
logs = trainer.state.log_history

# 학습 및 평가 데이터 추출
train_metrics = []
eval_metrics = []

for log in logs:
    if 'loss' in log: # 훈련 손실
        train_metrics.append({'epoch': log['epoch'], 'step': log['step'], 'metric': 'Loss', 'value': log['loss'], 'type': 'Train'})
    if 'eval_loss' in log: # 평가 손실
        eval_metrics.append({'epoch': log['epoch'], 'step': log['step'], 'metric': 'Loss', 'value': log['eval_loss'], 'type': 'Validation'})
    if 'eval_accuracy' in log: # 평가 정확도
        eval_metrics.append({'epoch': log['epoch'], 'step': log['step'], 'metric': 'Accuracy', 'value': log['eval_accuracy'], 'type': 'Validation'})

# DataFrame으로 변환
train_df = pd.DataFrame(train_metrics)
eval_df = pd.DataFrame(eval_metrics)

# 손실 그래프
plt.figure(figsize=(12, 6))
sns.lineplot(data=train_df[train_df['metric'] == 'Loss'], x='step', y='value', label='Train Loss')
sns.lineplot(data=eval_df[eval_df['metric'] == 'Loss'], x='step', y='value', label='Validation Loss')
plt.title('Training and Validation Loss Over Steps')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 정확도 그래프
plt.figure(figsize=(12, 6))
sns.lineplot(data=eval_df[eval_df['metric'] == 'Accuracy'], x='step', y='value', label='Validation Accuracy', color='green')
plt.title('Validation Accuracy Over Steps')
plt.xlabel('Steps')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


### 저장된 모델로 새로운 문장 감정 분석 테스트

In [ ]:
from transformers import pipeline

# 저장된 모델과 토크나이저 불러오기
loaded_tokenizer = AutoTokenizer.from_pretrained("./my_nsmc_bert_model")
loaded_model = AutoModelForSequenceClassification.from_pretrained("./my_nsmc_bert_model")

# 파이프라인 생성
# sentiment-analysis 파이프라인은 긍정/부정 분류에 사용됩니다.
classifier = pipeline("sentiment-analysis", model=loaded_model, tokenizer=loaded_tokenizer)

print("모델과 토크나이저 로드 및 파이프라인 생성 완료!")

In [ ]:
def predict_sentiment(text):
    result = classifier(text)[0]
    label = result['label']
    score = result['score']

    # 모델의 라벨이 'LABEL_0'과 'LABEL_1'로 되어있으므로, 이를 '부정'과 '긍정'으로 매핑
    sentiment_map = {'LABEL_0': '부정', 'LABEL_1': '긍정'}
    sentiment = sentiment_map.get(label, '알 수 없음')

    print(f"문장: '{text}'")
    print(f"예측 감정: {sentiment} (점수: {score:.4f})")

# 테스트할 문장들
sentences = [
    "이 영화 정말 재미있었어요! 강력 추천합니다.",
    "최악의 영화였다. 시간을 낭비했어.",
    "그냥 볼만했어요. 특별하진 않네요.",
    "생각보다 괜찮아서 놀랐습니다.",
    "다시는 안 볼 작품."
]

for sentence in sentences:
    predict_sentiment(sentence)


In [ ]:
while True:
    user_input = input("감정을 분석할 문장을 입력하세요 (종료하려면 'quit' 입력): ")
    if user_input.lower() == 'quit':
        print("감정 분석을 종료합니다.")
        break
    if user_input:
        predict_sentiment(user_input)
    else:
        print("문장을 입력해주세요.")